<a href="https://colab.research.google.com/github/Kaustubh484/DiffSplat/blob/claude%2Fcreate-jupyter-notebook-qMH6u/DiffSplat_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DiffSplat — Google Colab Demo

**Before running anything:**  
Go to `Runtime → Change runtime type` and select **GPU** (T4 is fine for SD1.5).

### What this notebook does
1. Clone this project and the official DiffSplat repo
2. Install all dependencies
3. Download pretrained checkpoints
4. Run text-conditioned 3D generation
5. Run image-conditioned 3D generation
6. Evaluate image quality (PSNR / SSIM / LPIPS)

> **Persistence**: Colab VMs are ephemeral. Mount Google Drive (optional cell below)
> to keep checkpoints across sessions and avoid re-downloading them each time.

## 0 — Check GPU

In [1]:
!nvidia-smi

Fri May  1 20:50:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
assert torch.cuda.is_available(), "No GPU found — go to Runtime → Change runtime type → GPU"
print(f"PyTorch {torch.__version__} | CUDA {torch.version.cuda} | GPU: {torch.cuda.get_device_name(0)}")

PyTorch 2.10.0+cu128 | CUDA 12.8 | GPU: Tesla T4


## 0b — (Optional) Mount Google Drive for persistent storage

Checkpoints are several GB. Mounting Drive lets you skip the download on future sessions.

In [3]:
USE_DRIVE = False  # Set True to mount Google Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSISTENT_ROOT = "/content/drive/MyDrive/DiffSplat"
else:
    PERSISTENT_ROOT = "/content"  # lost when the runtime disconnects

print(f"Persistent root: {PERSISTENT_ROOT}")

Persistent root: /content


## 1 — Configuration

In [4]:
import os, sys
from pathlib import Path

# ── Paths ────────────────────────────────────────────────────────────────────
WRAPPER_DIR = Path("/content/DiffSplat-wrapper")   # this project (wrapper)
REPO_DIR    = Path("/content/DiffSplat")            # official DiffSplat repo
OUT_DIR     = Path(PERSISTENT_ROOT) / "out"         # checkpoints + outputs
DATA_DIR    = Path(PERSISTENT_ROOT) / "data"        # benchmark data

# ── Model ────────────────────────────────────────────────────────────────────
# "sd15"  → Stable Diffusion 1.5  (~16 GB VRAM, works on T4)
# "pas"   → PixelArt-Sigma        (~16 GB VRAM)
# "sd35m" → SD 3.5 Medium         (~40 GB VRAM, needs A100)
MODEL = "sd15"

# ── PyTorch wheel channel ─────────────────────────────────────────────────────
# PyTorch 2.3.1 only publishes cu118 and cu121 wheels.
# cu121 wheels work on any CUDA 12.x driver (12.1, 12.4, 12.6, 12.8, etc.)
# because CUDA is backward-compatible. Do NOT try to auto-detect the driver
# version — it will produce a channel like cu128 that doesn't exist.
TORCH_CHANNEL = "cu121"

# ── Optional HF mirror ───────────────────────────────────────────────────────
HF_ENDPOINT = ""   # e.g. "https://hf-mirror.com" if hf.co is slow

GPU_ID = 0
SEED   = 0

print(f"WRAPPER_DIR   : {WRAPPER_DIR}")
print(f"REPO_DIR      : {REPO_DIR}")
print(f"OUT_DIR       : {OUT_DIR}")
print(f"MODEL         : {MODEL}")
print(f"TORCH_CHANNEL : {TORCH_CHANNEL}")

WRAPPER_DIR   : /content/DiffSplat-wrapper
REPO_DIR      : /content/DiffSplat
OUT_DIR       : /content/out
MODEL         : sd15
TORCH_CHANNEL : cu121


## 2 — Clone repos and install dependencies

This clones:
- **This wrapper project** (`kaustubh484/DiffSplat`) into `/content/DiffSplat-wrapper`
- **The official DiffSplat repo** into `/content/DiffSplat`

Then installs all Python dependencies. Takes 5–10 minutes on a fresh runtime.

In [5]:
import subprocess

def run(cmd, **kw):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True, **kw)

# Clone the wrapper (this project)
if not WRAPPER_DIR.exists():
    run(["git", "clone", "--depth", "1",
         "https://github.com/kaustubh484/DiffSplat.git",
         str(WRAPPER_DIR)])
else:
    print(f"Wrapper already exists at {WRAPPER_DIR}")

# Add wrapper to Python path so diffsplat_tools is importable
if str(WRAPPER_DIR) not in sys.path:
    sys.path.insert(0, str(WRAPPER_DIR))

print("Wrapper ready.")

$ git clone --depth 1 https://github.com/kaustubh484/DiffSplat.git /content/DiffSplat-wrapper
Wrapper ready.


In [6]:
from diffsplat_tools.setup import clone_repo, install_repo

# Clone the official DiffSplat repo
clone_repo(REPO_DIR, ref=None, dry_run=False)
print(f"Official repo ready at: {REPO_DIR}")

$ git clone --depth 1 https://github.com/chenguolin/DiffSplat.git /content/DiffSplat
Official repo ready at: /content/DiffSplat


In [ ]:
import subprocess, sys

pip = [sys.executable, "-m", "pip"]
torch_index = f"https://download.pytorch.org/whl/{TORCH_CHANNEL}"

def _run(cmd, cwd=None):
    print("$", " ".join(str(c) for c in cmd))
    subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True)

_run([*pip, "install", "-U", "pip", "setuptools", "wheel"])
_run([*pip, "install", "-U",
      "torch==2.3.1", "torchvision==0.18.1", "torchaudio==2.3.1",
      "--index-url", torch_index])
_run([*pip, "install", "-U", "xformers==0.0.27", "--index-url", torch_index])
_run([*pip, "install", "-U", "gpustat", "huggingface_hub", "scikit-image"])
_run([*pip, "install", "-U", "-r", "settings/requirements.txt"], cwd=REPO_DIR)

# --no-build-isolation is required on Python 3.12: distutils was removed, so
# packages that use setup.py break inside pip's isolated build environment.
# Passing this flag reuses the host setuptools which provides the shim.
#
# skip rasterizer (diff-gaussian-rasterization): CUDA C++ extension that must
# be compiled from source; Colab toolkit/driver mismatch breaks the build.
_run([*pip, "install", "--no-build-isolation",
      "-e", "extensions/diffusers_diffsplat"], cwd=REPO_DIR)

print("\nInstallation complete.")

In [ ]:
# Restart the Colab runtime so the newly installed packages are picked up.
# After the restart, run all cells from Section 3 onward (skip Sections 1–2).
import IPython
print("Restarting runtime to apply new packages...")
IPython.Application.instance().kernel.do_shutdown(True)

---
### ↑ Runtime restarts here — continue from this cell after the restart ↑
---

In [ ]:
# Re-run this after the restart to restore all variables
import sys, os, subprocess
from pathlib import Path
import torch

WRAPPER_DIR   = Path("/content/DiffSplat-wrapper")
REPO_DIR      = Path("/content/DiffSplat")
PERSISTENT_ROOT = "/content"   # change to Drive path if you mounted it
OUT_DIR       = Path(PERSISTENT_ROOT) / "out"
DATA_DIR      = Path(PERSISTENT_ROOT) / "data"
MODEL         = "sd15"
HF_ENDPOINT   = ""
GPU_ID        = 0
SEED          = 0

if str(WRAPPER_DIR) not in sys.path:
    sys.path.insert(0, str(WRAPPER_DIR))

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print("Ready.")

## 3 — Download Checkpoints

In [ ]:
from diffsplat_tools.common import DEFAULT_CACHE_ROOT, command_env
from diffsplat_tools.downloads import download_checkpoints

# Override cache dirs to use PERSISTENT_ROOT so they survive a Drive mount
hf_home    = str(Path(PERSISTENT_ROOT) / ".cache" / "huggingface")
torch_home = str(Path(PERSISTENT_ROOT) / ".cache" / "torch")

env = command_env(
    hf_home=hf_home,
    torch_home=torch_home,
    hf_endpoint=HF_ENDPOINT or None,
)

download_checkpoints(
    repo_dir=REPO_DIR,
    out_dir=OUT_DIR,
    model=MODEL,
    variant="both",
    env=env,
    dry_run=False,
)
print(f"Checkpoints in: {OUT_DIR}")

## 4 — Download T3Bench Prompts

In [ ]:
from diffsplat_tools.downloads import download_t3bench

download_t3bench(DATA_DIR, dry_run=False)
t3bench_file = DATA_DIR / "t3bench" / "t3bench_prompt.txt"
prompts = t3bench_file.read_text().strip().splitlines()
print(f"Loaded {len(prompts)} prompts. First 5:")
for p in prompts[:5]:
    print(" ", p)

## 5 — Text-Conditioned Inference

Generate a 3D Gaussian splat from a text prompt.

In [ ]:
TEXT_PROMPT       = "a_toy_robot"   # underscores instead of spaces
GUIDANCE_SCALE    = 7.5
NUM_TIMESTEPS     = 20
OUTPUT_VIDEO_TYPE = "gif"

cmd = [
    sys.executable,
    str(WRAPPER_DIR / "scripts" / "run_text.py"),
    "--model",                   MODEL,
    "--prompt",                  TEXT_PROMPT,
    "--seed",                    str(SEED),
    "--gpu-id",                  str(GPU_ID),
    "--guidance-scale",          str(GUIDANCE_SCALE),
    "--num-inference-timesteps", str(NUM_TIMESTEPS),
    "--output-video-type",       OUTPUT_VIDEO_TYPE,
    "--repo-dir",                str(REPO_DIR),
    "--out-dir",                 str(OUT_DIR),
]
if HF_ENDPOINT:
    cmd += ["--hf-endpoint", HF_ENDPOINT]

print("Running inference...")
subprocess.run(cmd, cwd=str(WRAPPER_DIR), check=True)

In [ ]:
import glob
from IPython.display import Image as IPImage, display
from diffsplat_tools.constants import MODEL_SPECS

tag = MODEL_SPECS[MODEL].text_tag
gifs = sorted(glob.glob(str(OUT_DIR / tag / f"*{TEXT_PROMPT}*.gif")))

if gifs:
    print(f"Result ({gifs[-1]}):")
    display(IPImage(filename=gifs[-1]))
else:
    print(f"No GIF found in {OUT_DIR / tag}")

## 6 — Image-Conditioned Inference

Reconstruct a 3D splat from a single reference image.

In [ ]:
# Use the bundled frog asset, or upload your own image to Colab and change this path
IMAGE_PATH    = REPO_DIR / "assets" / "grm" / "frog.png"
IMAGE_PROMPT  = "a_frog"
ELEVATION     = 20.0
IMG_GUIDANCE  = 2.0
IMG_TIMESTEPS = 20

# Preview the input
display(IPImage(filename=str(IMAGE_PATH), width=256))

In [ ]:
cmd = [
    sys.executable,
    str(WRAPPER_DIR / "scripts" / "run_image.py"),
    "--model",                   MODEL,
    "--image-path",              str(IMAGE_PATH),
    "--prompt",                  IMAGE_PROMPT,
    "--elevation",               str(ELEVATION),
    "--guidance-scale",          str(IMG_GUIDANCE),
    "--num-inference-timesteps", str(IMG_TIMESTEPS),
    "--rembg-and-center",
    "--triangle-cfg-scaling",
    "--seed",                    str(SEED),
    "--gpu-id",                  str(GPU_ID),
    "--repo-dir",                str(REPO_DIR),
    "--out-dir",                 str(OUT_DIR),
]
if HF_ENDPOINT:
    cmd += ["--hf-endpoint", HF_ENDPOINT]

subprocess.run(cmd, cwd=str(WRAPPER_DIR), check=True)

In [ ]:
img_tag = MODEL_SPECS[MODEL].image_tag
gifs = sorted(glob.glob(str(OUT_DIR / img_tag / f"*{IMAGE_PROMPT}*.gif")))

if gifs:
    print(f"Result ({gifs[-1]}):")
    display(IPImage(filename=gifs[-1]))
else:
    print(f"No GIF found in {OUT_DIR / img_tag}")

## 7 — Upload Your Own Image (optional)

Use Colab's file upload widget to bring in a custom image, then re-run inference.

In [ ]:
from google.colab import files

uploaded = files.upload()   # opens the file picker

if uploaded:
    fname = next(iter(uploaded))
    custom_image = Path("/content") / fname
    display(IPImage(filename=str(custom_image), width=256))
    print(f"Uploaded: {custom_image}")

    # Run image-conditioned inference on the uploaded image
    cmd = [
        sys.executable,
        str(WRAPPER_DIR / "scripts" / "run_image.py"),
        "--model",     MODEL,
        "--image-path", str(custom_image),
        "--prompt",    "",            # leave empty or describe the object
        "--elevation", "20",
        "--rembg-and-center",
        "--triangle-cfg-scaling",
        "--repo-dir",  str(REPO_DIR),
        "--out-dir",   str(OUT_DIR),
    ]
    subprocess.run(cmd, cwd=str(WRAPPER_DIR), check=True)

    # Show output
    result_gifs = sorted(glob.glob(str(OUT_DIR / img_tag / "*.gif")))
    if result_gifs:
        display(IPImage(filename=result_gifs[-1]))

## 8 — Image Quality Evaluation (PSNR / SSIM / LPIPS)

Requires a directory of predicted renders and matching ground-truth images.

In [ ]:
import json, argparse
from diffsplat_tools.evaluation import evaluate_image_dirs

PRED_DIR     = OUT_DIR / "predictions"   # change to your predicted renders
GT_DIR       = DATA_DIR / "gso_rendered" # change to your ground-truth dir
METRICS_JSON = Path("/content") / "metrics.json"

if not PRED_DIR.exists() or not GT_DIR.exists():
    print("Set PRED_DIR and GT_DIR to existing directories to run evaluation.")
    print(f"  PRED_DIR : {PRED_DIR}  exists={PRED_DIR.exists()}")
    print(f"  GT_DIR   : {GT_DIR}  exists={GT_DIR.exists()}")
else:
    eval_args = argparse.Namespace(
        pred_dir=str(PRED_DIR),
        gt_dir=str(GT_DIR),
        json=str(METRICS_JSON),
        device="auto",
        skip_lpips=False,
        save_per_image=True,
        limit=None,
    )
    evaluate_image_dirs(eval_args)

    if METRICS_JSON.exists():
        m = json.loads(METRICS_JSON.read_text())
        print(f"\nImages : {m['count']}")
        print(f"PSNR   : {m['psnr']['mean']:.2f} ± {m['psnr']['std']:.2f} dB")
        print(f"SSIM   : {m['ssim']['mean']:.4f} ± {m['ssim']['std']:.4f}")
        if 'lpips' in m:
            print(f"LPIPS  : {m['lpips']['mean']:.4f} ± {m['lpips']['std']:.4f}")

## Colab Tips

| Problem | Fix |
|---------|-----|
| Session disconnects and you lose checkpoints | Mount Google Drive (Section 0b) and set `PERSISTENT_ROOT` to your Drive path |
| `RuntimeError: CUDA out of memory` | Use `sd15` model; set `--half-precision` flag |
| Colab CUDA version mismatch | The `TORCH_CHANNEL` is auto-detected from `torch.version.cuda` |
| Slow Hugging Face downloads | Set `HF_ENDPOINT = "https://hf-mirror.com"` |
| Want to use `sd35m` | Upgrade to Colab Pro and select A100 GPU |
| Packages not found after install | The runtime-restart cell (Section 2) handles this |

To add `--half-precision` for lower VRAM usage, append it to any `cmd` list:
```python
cmd.append("--half-precision")
```